In [ ]:
# Fase 3 baseline: duck harness (Tufa Labs) en la G4 — validación offline CORTA.
import json, os, pickle, subprocess, sys, sysconfig, time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1","true"}
NOTEBOOK_START = time.time()
os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
os.environ["ONLY_RESET_LEVELS"] = "true"
# CUDA linker path para vLLM/torch en imagen Kaggle
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    e for e in ["/usr/local/nvidia/lib64", os.environ.get("LIBRARY_PATH","")] if e)
# Corte de validación offline (minutos) para NO gastar 9h de G4 cuando no es rerun.
OFFLINE_SOFT_MIN = float(os.environ.get("TAAF_OFFLINE_SOFT_MIN", "25"))
WORKING = Path("/kaggle/working"); WORKING.mkdir(parents=True, exist_ok=True)
print("TRUE_SUBMISSION =", TRUE_SUBMISSION)


In [ ]:
# Instalar arc-agi del wheelhouse de la competencia (offline)
COMP_ROOT = None
for dp, dn, _ in os.walk("/kaggle/input"):
    if "arc_agi_3_wheels" in dn:
        COMP_ROOT = Path(dp); break
assert COMP_ROOT, "wheelhouse no encontrado"
subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--no-index",
    "--no-warn-conflicts","--disable-pip-version-check",
    f"--find-links={COMP_ROOT/'arc_agi_3_wheels'}","arc-agi"], stdout=subprocess.DEVNULL)
import arc_agi; print("arc_agi OK")

# Localizar el bundle del solver por su marker
BUNDLE = None
for m in Path("/kaggle/input").rglob("taaf-kaggle-bundle.json"):
    BUNDLE = m.parent; break
assert BUNDLE, "bundle TAAF no encontrado (adjunta thtennant/taaf-kaggle-source-share-fork)"
print("BUNDLE =", BUNDLE)

# Mapear datasets adjuntos a sus mounts
DATASET_SOURCES = ["thtennant/taaf-kaggle-source-share-fork",
                   "driessmit1/arc3-vllm-h100-wheelhouse-v3",
                   "driessmit1/vrfai-qwen3-6-27b-fp8-hf-snapshot"]
def mount(ref):
    o,s = ref.split("/",1)
    for c in (Path("/kaggle/input")/s, Path("/kaggle/input/datasets")/o/s):
        if c.exists(): return str(c)
    return str(Path("/kaggle/input")/s)
paths = {r: (str(BUNDLE) if i==0 else mount(r)) for i,r in enumerate(DATASET_SOURCES)}
env_extra = {"TAAF_KAGGLE_INPUT_PATHS": json.dumps(paths, sort_keys=True),
             "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
             "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps([])}
os.environ.update(env_extra)
SETUP_ENV = WORKING/"taaf_setup_env.json"; SETUP_ENV.write_text(json.dumps(env_extra))
print(paths)


In [ ]:
# Importar repos del bundle y correr setup_commands (instala vLLM, arranca el server)
def source_entries(b):
    out=[]
    for repo in sorted((b/"src").iterdir(), reverse=True):
        for c in (repo/"src", repo):
            if c.is_dir(): out.append(c)
    return out
entries = source_entries(BUNDLE)
for e in entries: sys.path.insert(0, str(e))
pth = Path(sysconfig.get_paths()["purelib"])/"taaf_sources.pth"
pth.write_text("".join(f"{e}\n" for e in entries))

def cmd_env():
    env = os.environ.copy(); env["PYTHON"]=sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"]=str(BUNDLE); env["TAAF_KAGGLE_WORKING_DIR"]=str(WORKING)
    env["TAAF_KAGGLE_SETUP_ENV"]=str(SETUP_ENV)
    env.update({str(k):str(v) for k,v in json.loads(SETUP_ENV.read_text()).items()})
    return env
env = cmd_env()
for c in json.loads((BUNDLE/"setup_commands.json").read_text()):
    print("setup:", c[:80], flush=True)
    subprocess.run(c, shell=True, check=True, cwd=WORKING, env=env)
    env = cmd_env(); os.environ.update(env)
for e in reversed([x for x in os.environ.get("PYTHONPATH","").split(os.pathsep) if x]):
    if e not in sys.path: sys.path.insert(0, e)
print("setup completo")


In [ ]:

# ---- NUESTRO BANCO contra el 27B servido por vLLM en localhost:1234 ----
import json, re, time, urllib.request

RAW = "https://raw.githubusercontent.com/jvilladuque90/arc-prize-2026-arc-agi-3/main"
MODELO = "vrfai/Qwen3.6-27B-FP8"      # kaggle_served_model_name del solver

def bajar(p):
    with urllib.request.urlopen(f"{RAW}/{p}?cb={int(time.time())}", timeout=60) as r:
        return r.read().decode("utf-8")

ITEMS = [json.loads(l) for l in bajar("micro_bench.jsonl").splitlines() if l.strip()]
_mp = {}
exec(compile(bajar("scripts/micro_prompts.py"), "micro_prompts.py", "exec"), _mp)
VARIANTS, normalize = _mp["VARIANTS"], _mp["normalize"]
trivial_baselines, paired_contrast = _mp["trivial_baselines"], _mp["paired_contrast"]
print(f"banco: {len(ITEMS)} items", flush=True)

# esperar a que el servidor este vivo (el setup ya lo arranco)
BASE = "http://127.0.0.1:1234/v1"
dl = time.monotonic() + 900
while time.monotonic() < dl:
    try:
        with urllib.request.urlopen(f"{BASE}/models", timeout=5) as r:
            if r.status == 200:
                break
    except Exception:
        pass
    time.sleep(10)
print("vLLM listo", flush=True)

def preguntar(prompt, thinking, max_tokens=64):
    cuerpo = json.dumps({
        "model": MODELO,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0, "max_tokens": max_tokens,
        "chat_template_kwargs": {"enable_thinking": bool(thinking)},
    }).encode()
    req = urllib.request.Request(f"{BASE}/chat/completions", data=cuerpo,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=300) as r:
        d = json.loads(r.read())
    m = d["choices"][0]["message"]
    return m.get("content") or "", d.get("usage", {}).get("completion_tokens", 0)

BASES = trivial_baselines(ITEMS)
print("base trivial:", json.dumps(BASES, ensure_ascii=False), flush=True)

# Se mide con thinking OFF y ON: ademas de la precision del 27B, da su coste real
# en tokens sobre las MISMAS preguntas donde el 4B saco 90.9%.
resultado = {"modelo": MODELO, "variantes": {}}
for pensar in (False, True):
    for nombre, tipo, constructor in VARIANTS:
        if not nombre.startswith(("A.", "B.V3", "C.")):
            continue                      # los mas informativos, para no eternizar
        sub = [i for i in ITEMS if i["type"] == tipo]
        if not sub:
            continue
        aciertos, toks, ejem = 0, [], []
        for it in sub:
            try:
                txt, nt = preguntar(constructor(it), pensar,
                                    max_tokens=512 if pensar else 64)
            except Exception as exc:
                txt, nt = f"<error {type(exc).__name__}>", 0
            toks.append(nt)
            ok = normalize(txt, tipo) == it["answer"]
            aciertos += ok
            if len(ejem) < 2:
                ejem.append({"esperado": it["answer"], "crudo": txt.strip()[:120]})
        clave = f"{nombre}{'_think' if pensar else ''}"
        resultado["variantes"][clave] = {
            "n": len(sub), "aciertos": aciertos,
            "precision": round(aciertos / len(sub), 3),
            "tokens_medios": round(sum(toks) / max(1, len(toks)), 1),
            "ejemplos": ejem}
        print(f"  {clave:22} {aciertos:3}/{len(sub):3} = {aciertos/len(sub):6.1%} | "
              f"{resultado['variantes'][clave]['tokens_medios']:7.1f} tok", flush=True)

print("\n===== BENCH 27B =====")
print(json.dumps(resultado, indent=2, ensure_ascii=False)[:6000])
